# Day 04: Smart Hygiene Risk Prediction System
**Machine Learning Classification Pipeline**
This notebook demonstrates an end-to-end Machine Learning pipeline to predict facility hygiene risk (`Low`, `Medium`, `High`) based on operational telemetry.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load cleaned dataset
data_path = os.path.join("..", "dataset", "processed", "facility_hygiene_cleaned.csv")
df = pd.read_csv(data_path)
print(f"Loaded clean dataset shape: {df.shape}")
df.head()

## 1. Exploratory Data Analysis (EDA)
We examine class balance across `hygiene_risk` and check correlation between numerical features.

In [2]:
# Class distribution
print("Class Distribution:")
print(df["hygiene_risk"].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="hygiene_risk", order=["Low", "Medium", "High"], palette=["#43A047", "#FB8C00", "#E53935"])
plt.title("Target Class Distribution: hygiene_risk")
plt.show()

## 2. Preprocessing & Pipeline Construction
We define a `ColumnTransformer` that imputes and standardizes numerical features, while one-hot encoding categorical features.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

num_features = ["cleanliness_score", "odor_score", "complaints", "footfall", "hours_since_cleaning"]
cat_features = ["waste_level"]
target = "hygiene_risk"

X = df[num_features + cat_features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_features),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_features)
])
print("Preprocessor defined successfully.")

## 3. Model Training & Comparison
We train both `LogisticRegression` and `RandomForestClassifier` on the training set.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

lr_pipe = Pipeline([("prep", preprocessor), ("clf", LogisticRegression(max_iter=1000, random_state=42))])
rf_pipe = Pipeline([("prep", preprocessor), ("clf", RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42))])

lr_pipe.fit(X_train, y_train)
rf_pipe.fit(X_train, y_train)

lr_pred = lr_pipe.predict(X_test)
rf_pred = rf_pipe.predict(X_test)

print(f"Logistic Regression Accuracy : {accuracy_score(y_test, lr_pred)*100:.2f}%")
print(f"Random Forest Classifier Accuracy: {accuracy_score(y_test, rf_pred)*100:.2f}%")

## 4. Model Evaluation & Confusion Matrix
We inspect precision, recall, and the confusion matrix for the chosen model.

In [5]:
from sklearn.metrics import confusion_matrix
labels = ["Low", "Medium", "High"]
cm = confusion_matrix(y_test, lr_pred, labels=labels)

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.title("Confusion Matrix - Logistic Regression")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

print("
Detailed Classification Report:")
print(classification_report(y_test, lr_pred, labels=labels))

## 5. Live Inference Demonstration
We test single facility predictions with probability confidence scores.

In [6]:
sample_input = pd.DataFrame([{
    "cleanliness_score": 94.0,
    "odor_score": 1,
    "waste_level": "Low",
    "complaints": 0,
    "footfall": 800,
    "hours_since_cleaning": 1.5
}])

pred_risk = lr_pipe.predict(sample_input)[0]
probs = lr_pipe.predict_proba(sample_input)[0]
print(f"Predicted Risk: {pred_risk}")
for cls_name, prob in zip(lr_pipe.classes_, probs):
    print(f"  {cls_name}: {prob*100:.1f}%")

## 6. Summary & Conclusion
- **Winner Model:** Logistic Regression achieved ~95.8% test accuracy and balanced macro F1 score.
- **Key Drivers:** `cleanliness_score` and `odor_score` are dominant linear features, while `hours_since_cleaning` acts as a strong risk multiplier.